# 85. 银行客户营销转化分析

<!-- module-learning-arc:start -->
> **综合项目 模块主线｜第 4 / 4 步：从有限资源走向优先级决策**
>
> **持续应用背景：** 进入数据分析决策实验室：连续处理客户价值、物流履约、供需调度和营销资源四类问题，训练从业务问题到行动建议的迁移能力。
>
> **承接上一阶段：** 共享单车需求与运力调度  →  **本章任务：** 银行客户营销转化分析  →  **下一步：** 模块大作业《跨模块业务决策项目》
>
> **大作业连接：** 本章练习将成为《跨模块业务决策项目》的一部分，最终需要把前四个项目形成的方法迁移为项目提案、最短充分证据链和决策备忘录。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：银行每天都要决定把有限的外呼资源投给哪批客户，这一章要处理的便是一份真实的银行电话营销记录（UCI Bank Marketing，4 万多条），每条记录都标出了客户特征、本次触达方式，以及最终是否认购定期存款。我们要回答的核心问题是：名单有限时，该先给谁打电话、又能覆盖多少愿意转化的客户。学完这一节，你就具备把一份营销历史数据清洗并整理成决策依据的完整思路。



## 本章目标

学完本章，你将能够：

- **理解**：理解「银行客户营销转化分析」的核心概念、适用场景与关键口径。
- **操作**：能按本章步骤写出可复现的实现，并读懂输出/结果。
- **迁移**：能用本章方法处理一份新数据，独立完成同类任务并给出结论。


## 85.1 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| age/job/education | 客户画像 | 人口与职业类别 |
| housing/loan/default | 信贷状态 | 含 unknown |
| contact/month/day_of_week | 触达渠道与时间 | 活动字段 |
| duration | 本次通话时长 | 强泄漏，名单生成时未知 |
| campaign/pdays/previous | 联系历史 | 999 表示此前未联系 |
| poutcome | 上次活动结果 | 历史信号 |
| y | 是否认购定期存款 | 目标 |

## 85.2 数据质量检查清单

- 分隔符与字段类型
- unknown 的分布
- 目标类别不平衡
- duration 泄漏
- campaign 极端重复触达
- 时间字段不是完整时间戳


## 85.3 项目任务

1. 加载并审计
2. 分析转化与触达疲劳
3. 构建呼叫前模型
4. 评价 ROC-AUC、PR-AUC 与召回率
5. 计算 Top 10% lift
6. 输出合规营销建议


## 85.4 项目阶段速查

先看每个阶段要做什么、留下什么证据，再按任务顺序运行项目代码。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 加载与营销质量审计 | `pd.read_csv()`、`df.astype()`、`df.target.mean()`、`unknown.head()` | unknown 是原数据的显式类别，不擅自当作真实的“否”；先量化再决定处理。 | 分隔符与字段类型 |
| 2. 客群与触达诊断 | `df.groupby()`、`pd.cut()`、`job.round()`、`fatigue.round()` | 这是描述性比较；活动名单本身存在选择机制，不能把组间差异解释为干预效果。 | unknown 的分布 |
| 3. 构建无事后泄漏模型 | `X_train.select_dtypes()`、`columns.tolist()`、`.fit()`、`df[features]` | 排除 duration，因为生成呼叫名单时本次通话尚未发生；使用分层切分和类别权重。 | 目标类别不平衡 |
| 4. 分类性能与Top-K Lift | `model.predict_proba()`、`pd.Series()`、`pd.DataFrame()`、`y_test.to_numpy()` | 营销名单关心有限容量内能覆盖多少转化客户，因此 PR-AUC 和 Top-K lift 比单独准确率更有用。 | duration 泄漏 |
| 5. 营销策略与治理 | `ranked.head()`、`top10.y.mean()`、`int()` | 模型排序不能替代客户同意、频控和成本收益规则。 | campaign 极端重复触达 |


## 85.5 项目交付物

- 一份可复现的分析 Notebook
- 清洗规则与关键指标表
- 至少一张支持结论的图表
- 结论、限制和下一步建议

## 85.6 阶段检查点

- [ ] 问题和数据字典完成
- [ ] 质量检查和清洗记录完成
- [ ] 核心指标或图表完成
- [ ] 结论与限制完成

## 85.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 85.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 85.9 加载与营销质量审计

unknown 是原数据的显式类别，不擅自当作真实的“否”；先量化再决定处理。


<!-- math-foundation:chapter-85 -->
### 数学推导｜营销名单的转化率与 Lift

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜计算总体基准率。** $p_{all}=P/N$，其中 $P$ 是全部样本中的正类数。

**第 2 步｜按模型分数取前 $K$ 名。** 名单命中数为 $TP_K$，名单转化率 $p_K=TP_K/K$。

**第 3 步｜与随机名单比较。** 

$$
Lift@K=\frac{p_K}{p_{all}}
$$

Lift 为 2 表示该名单的正类浓度约为总体的 2 倍，不表示转化人数翻倍，也不等于利润翻倍。

**把上面的关系收束为本章计算式：**

$$
conversion=\frac{TP}{N_{contact}},\qquad Lift@K=\frac{conversion@K}{conversion_{all}}
$$

**符号解释：** $TP$ 是成功转化人数，$K$ 是按模型分数选出的名单规模。

**代码对应：** 按概率降序取 Top-K，比较名单转化率、覆盖率和总体基准。

**使用边界：** Lift 不包含联系成本和客户价值；模型名单仍需合规与公平性检查。


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("/datasets/bank_marketing_full.csv", sep=";")
df["target"] = (df.y == "yes").astype(int)
unknown = (df.astype(str) == "unknown").sum().sort_values(ascending=False)
print("形状:", df.shape, " 转化率:", f"{df.target.mean():.2%}")
print("unknown最多字段:\n", unknown.head(8))
print(
    "单次活动联系次数:\n",
    df.campaign.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1),
)


**练一练**：质量审计不能只下结论，先量化数据再决定怎么处理。回到上面加载出的银行营销数据 `df`，请分别统计 `housing`、`loan`、`default` 三个信贷字段里显式缺失值 `unknown` 各有多少行，并把结果存放在字典 `unknown_counts` 中打印出来。

> 提示：可通过 `df["housing"] == "unknown"` 的布尔比较配 `.sum()` 得到计数。若无法读取原始 CSV，可先用下面这条代码生成同名小样本数据再练习：
> ```python
> df = pd.DataFrame({"housing": ["yes","no","unknown"], "loan": ["unknown","no","yes"], "default": ["no","unknown","unknown"]})
> ```


In [ ]:
# 请在下方填写代码
# 目标：统计 df 中 housing、loan、default 三个字段里 "unknown" 的行数，结果存入字典并打印
# 提示：可通过 df[col] == "unknown" 的布尔比较配 .sum() 得到某字段的 unknown 行数
# TODO：请在下方完成 —— 练一练：质量审计不能只下结论，先量化数据再决定怎么处理。回到上面加载出的银行营销数据 df，请分别统计 housing、


In [ ]:
# 分别统计三个信贷字段中显式缺失值 unknown 的分布
unknown_counts = {
    c: int((df[c] == "unknown").sum()) for c in ["housing", "loan", "default"]
}
print(unknown_counts)


## 85.10 客群与触达诊断

这是描述性比较；活动名单本身存在选择机制，不能把组间差异解释为干预效果。


In [ ]:
job = (
    df.groupby("job")
    .agg(customers=("target", "size"), conversion=("target", "mean"))
    .query("customers>=200")
    .sort_values("conversion", ascending=False)
)
touch = pd.cut(
    df.campaign,
    [0, 1, 2, 3, 5, 10, np.inf],
    labels=["1", "2", "3", "4-5", "6-10", "11+"],
)
fatigue = df.groupby(touch, observed=True).agg(
    customers=("target", "size"), conversion=("target", "mean")
)
print("职业客群:\n", job.round(3))
print("联系次数与转化:\n", fatigue.round(3))
print("注意：低意向客户可能被多次联系，不能据此断言多联系导致低转化。")


## 85.11 构建无事后泄漏模型

排除 duration，因为生成呼叫名单时本次通话尚未发生；使用分层切分和类别权重。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

features = [c for c in df.columns if c not in ["y", "target", "duration"]]
X_train, X_test, y_train, y_test = train_test_split(
    df[features],
    df.target,
    test_size=0.25,
    stratify=df.target,
    random_state=75,
)
cat = X_train.select_dtypes(include="object").columns.tolist()
num = [c for c in features if c not in cat]
prep = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ("num", StandardScaler(), num),
    ]
)
model = Pipeline(
    [
        ("prep", prep),
        (
            "model",
            LogisticRegression(
                max_iter=700, class_weight="balanced", random_state=75
            ),
        ),
    ]
).fit(X_train, y_train)
print("已排除事后字段 duration；训练/测试:", len(X_train), len(X_test))


## 85.12 分类性能与Top-K Lift

营销名单关心有限容量内能覆盖多少转化客户，因此 PR-AUC 和 Top-K lift 比单独准确率更有用。


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
)

prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)
print(
    pd.Series(
        {
            "ROC-AUC": roc_auc_score(y_test, prob),
            "PR-AUC": average_precision_score(y_test, prob),
            "precision@0.5": precision_score(y_test, pred),
            "recall@0.5": recall_score(y_test, pred),
        }
    ).round(3)
)
ranked = pd.DataFrame({"y": y_test.to_numpy(), "p": prob}).sort_values(
    "p", ascending=False
)
for share in [0.05, 0.10, 0.20]:
    top = ranked.head(int(len(ranked) * share))
    lift = top.y.mean() / ranked.y.mean()
    print(f"Top {share:.0%}: 名单 {len(top)}")
    print(f"  转化率 {top.y.mean():.2%}，lift={lift:.2f}")
    print(f"  覆盖转化 {top.y.sum() / ranked.y.sum():.1%}")


## 85.13 营销策略与治理

模型排序不能替代客户同意、频控和成本收益规则。


In [ ]:
top10 = ranked.head(int(len(ranked) * 0.1))
print(f"1. 若容量为测试客户的10%，模型名单转化率约 {top10.y.mean():.2%}，上线前需用新活动做随机对照验证增量。")
print("2. 对 campaign 设置频控并监控退订/投诉；不能从观察数据断言重复联系的因果伤害。")
print("3. 上线监控 PR-AUC、Top-K转化、覆盖率、不同客户群的触达率与投诉率。")
print("限制：数据来自历史电话活动，缺少完整成本、同意状态和时间戳；模型预测相关性，不预测营销的个体因果增量。")


## 85.14 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 85.14.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 85.14.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 85.15 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 85.15.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 85.16 易错点提醒

**易错点 1**：该数据用分号分隔（sep=";"），read_csv 不指定分隔符会读成一整列。

**易错点 2**：正类（认购）占比很低，直接看准确率会"假阳性"；用混淆矩阵和转化率对比。

**易错点 3**：同一客户可能被多次营销（previous/pdays 记录历史接触），统计触达次数时别把同一客户重复计。

**易错点 4**：duration（通话时长）在通话结束后才知道，做预测特征时它是泄漏；分析时要区分"通话后统计"与"通话前预测"。

**易错点 5**：缺失值（如 pdays=999 表示从未联系）是业务编码不是脏数据，处理前先查数据字典。


## 85.17 结论与表达

- 呼叫前模型必须排除 duration。
- 不平衡营销任务应报告 PR-AUC 与容量相关 lift。
- 高响应概率不等于高增量响应。
- 频控、同意与公平触达属于部署必要条件。


## 85.18 项目验收清单

- 正确读取分号 CSV
- 明确排除 duration
- 能计算 Top 10% lift
- 能区分响应模型与 uplift/因果模型

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 85.19 小结

使用 UCI Bank Marketing 41,188 条真实电话营销记录，分析客户触达与定期存款转化，建立避免通话时长泄漏的营销评分基线。


### 85.19.1 你已经完成

- 处理分号分隔与 unknown 类别
- 分析触达次数和客户结构
- 识别 duration 的事后泄漏
- 在类别不平衡下评价模型
- 按有限呼叫容量输出 lift 分层


### 85.19.2 质量与结论提醒

- 分隔符与字段类型
- unknown 的分布
- 目标类别不平衡
- 呼叫前模型必须排除 duration。
- 不平衡营销任务应报告 PR-AUC 与容量相关 lift。
- 高响应概率不等于高增量响应。
- 频控、同意与公平触达属于部署必要条件。


### 85.19.3 项目交付检查

- [ ] 正确读取分号 CSV
- [ ] 明确排除 duration
- [ ] 能计算 Top 10% lift
- [ ] 能区分响应模型与 uplift/因果模型


### 85.19.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
